In [ ]:
!pip -q install datasets transformers evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [ ]:
import re
import time
import random
import numpy as np
import pandas as pd
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed
)
import evaluate

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


# 1. Load dataset

In [ ]:
dataset = load_dataset("emotion")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [ ]:
print(dataset["train"][0])
print(dataset["train"].features)

{'text': 'i didnt feel humiliated', 'label': 0}
{'text': Value('string'), 'label': ClassLabel(names=['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'])}


In [ ]:
label_names = dataset["train"].features["label"].names
num_classes = len(label_names)

print("Label names:", label_names)
print("Number of classes:", num_classes)

Label names: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
Number of classes: 6


In [ ]:
train_texts = dataset["train"]["text"]
val_texts   = dataset["validation"]["text"]
test_texts  = dataset["test"]["text"]

train_labels = dataset["train"]["label"]
val_labels   = dataset["validation"]["label"]
test_labels  = dataset["test"]["label"]

print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))
print("Test size:", len(test_texts))

Train size: 16000
Validation size: 2000
Test size: 2000


# 2. CNN approach
Clean text, build vocabulary, convert sentences to token id sequences, pad to same length, and use TextCNN for classification.

## 2.1 Preprocessing for CNN

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text):
    return text.split()

train_texts_cnn = [clean_text(t) for t in train_texts]
val_texts_cnn   = [clean_text(t) for t in val_texts]
test_texts_cnn  = [clean_text(t) for t in test_texts]

print(train_texts_cnn[0])

i didnt feel humiliated


## 2.2 Create vocabulary

In [ ]:
counter = Counter()
for text in train_texts_cnn:
    counter.update(tokenize(text))

max_vocab_size = 20000
special_tokens = ["<PAD>", "<UNK>"]

most_common = counter.most_common(max_vocab_size - len(special_tokens))
vocab = {tok: idx for idx, tok in enumerate(special_tokens)}

for word, _ in most_common:
    vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))
print("Sample vocab items:", list(vocab.items())[:10])

Vocabulary size: 15214
Sample vocab items: [('<PAD>', 0), ('<UNK>', 1), ('i', 2), ('feel', 3), ('and', 4), ('to', 5), ('the', 6), ('a', 7), ('feeling', 8), ('that', 9)]


## 2.3 Encode and Padding

In [ ]:
def encode_text(text, vocab):
    return [vocab.get(tok, vocab["<UNK>"]) for tok in tokenize(text)]

train_encoded = [encode_text(t, vocab) for t in train_texts_cnn]
val_encoded   = [encode_text(t, vocab) for t in val_texts_cnn]
test_encoded  = [encode_text(t, vocab) for t in test_texts_cnn]

lengths = [len(x) for x in train_encoded]
max_len = min(50, max(lengths))

print("Max sequence length for CNN:", max_len)
print("Min length:", min(lengths))
print("Mean length:", round(sum(lengths) / len(lengths), 2))

Max sequence length for CNN: 50
Min length: 2
Mean length: 19.17


In [ ]:
def pad_sequence(seq, max_len, pad_value=0):
    if len(seq) < max_len:
        seq = seq + [pad_value] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
    return seq

train_padded = [pad_sequence(x, max_len) for x in train_encoded]
val_padded   = [pad_sequence(x, max_len) for x in val_encoded]
test_padded  = [pad_sequence(x, max_len) for x in test_encoded]

## 2.4 Dataset và DataLoader

In [ ]:
class EmotionCNNDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = torch.tensor(texts, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

train_ds_cnn = EmotionCNNDataset(train_padded, train_labels)
val_ds_cnn   = EmotionCNNDataset(val_padded, val_labels)
test_ds_cnn  = EmotionCNNDataset(test_padded, test_labels)

train_loader_cnn = DataLoader(train_ds_cnn, batch_size=64, shuffle=True)
val_loader_cnn   = DataLoader(val_ds_cnn, batch_size=64)
test_loader_cnn  = DataLoader(test_ds_cnn, batch_size=64)

## 2.5 Build Model TextCNN

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, filter_sizes=[3,4,5], num_filters=100, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)

        conv_outs = []
        for conv in self.convs:
            c = F.relu(conv(x))
            p = F.max_pool1d(c, kernel_size=c.shape[2]).squeeze(2)
            conv_outs.append(p)

        out = torch.cat(conv_outs, dim=1)
        out = self.dropout(out)
        out = self.fc(out)
        return out

## 2.6 Training and Evaluation for CNN

In [ ]:
def train_one_epoch_cnn(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

@torch.no_grad()
def evaluate_cnn(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")
    return total_loss / len(loader), acc, f1, all_labels, all_preds

In [ ]:
cnn_model = TextCNN(
    vocab_size=len(vocab),
    embed_dim=128,
    num_classes=num_classes,
    filter_sizes=[3,4,5],
    num_filters=100,
    dropout=0.5
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=1e-3)

num_epochs_cnn = 8
best_val_acc_cnn = 0
cnn_history = []

start_time_cnn = time.time()

for epoch in range(num_epochs_cnn):
    train_loss = train_one_epoch_cnn(cnn_model, train_loader_cnn, optimizer, criterion, device)
    val_loss, val_acc, val_f1, _, _ = evaluate_cnn(cnn_model, val_loader_cnn, criterion, device)

    cnn_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_f1_macro": val_f1
    })

    print(f"Epoch {epoch+1}/{num_epochs_cnn}")
    print(f"  Train loss: {train_loss:.4f}")
    print(f"  Val loss:   {val_loss:.4f}")
    print(f"  Val acc:    {val_acc:.4f}")
    print(f"  Val f1:     {val_f1:.4f}")

    if val_acc > best_val_acc_cnn:
        best_val_acc_cnn = val_acc
        torch.save(cnn_model.state_dict(), "best_textcnn.pt")

cnn_train_time = time.time() - start_time_cnn
print(f"\nCNN training time: {cnn_train_time:.2f} seconds")

Epoch 1/8
  Train loss: 1.5752
  Val loss:   1.3304
  Val acc:    0.5190
  Val f1:     0.2991
Epoch 2/8
  Train loss: 1.1245
  Val loss:   0.7728
  Val acc:    0.7245
  Val f1:     0.6280
Epoch 3/8
  Train loss: 0.6065
  Val loss:   0.4302
  Val acc:    0.8610
  Val f1:     0.8269
Epoch 4/8
  Train loss: 0.3505
  Val loss:   0.3361
  Val acc:    0.8820
  Val f1:     0.8543
Epoch 5/8
  Train loss: 0.2336
  Val loss:   0.2911
  Val acc:    0.8980
  Val f1:     0.8690
Epoch 6/8
  Train loss: 0.1692
  Val loss:   0.2807
  Val acc:    0.9035
  Val f1:     0.8787
Epoch 7/8
  Train loss: 0.1378
  Val loss:   0.2722
  Val acc:    0.9055
  Val f1:     0.8798
Epoch 8/8
  Train loss: 0.1104
  Val loss:   0.2846
  Val acc:    0.9050
  Val f1:     0.8806

CNN training time: 12.04 seconds


In [ ]:
cnn_history_df = pd.DataFrame(cnn_history)
cnn_history_df

,epoch,train_loss,val_loss,val_accuracy,val_f1_macro
0,1,1.575177,1.330436,0.5190,0.299062
1,2,1.124481,0.772830,0.7245,0.628021
2,3,0.606492,0.430152,0.8610,0.826852
3,4,0.350459,0.336098,0.8820,0.854252
4,5,0.233563,0.291069,0.8980,0.869044
5,6,0.169168,0.280749,0.9035,0.878704
6,7,0.137780,0.272185,0.9055,0.879829
7,8,0.110384,0.284562,0.9050,0.880597


## 2.7 Validation and Test for CNN

In [ ]:
cnn_model.load_state_dict(torch.load("best_textcnn.pt", map_location=device))

cnn_test_loss, cnn_test_acc, cnn_test_f1, cnn_y_true, cnn_y_pred = evaluate_cnn(
    cnn_model, test_loader_cnn, criterion, device
)

print("CNN Test loss:", round(cnn_test_loss, 4))
print("CNN Test accuracy:", round(cnn_test_acc, 4))
print("CNN Test macro F1:", round(cnn_test_f1, 4))

CNN Test loss: 0.2736
CNN Test accuracy: 0.9025
CNN Test macro F1: 0.8605


In [ ]:
print(classification_report(cnn_y_true, cnn_y_pred, target_names=label_names))

              precision    recall  f1-score   support

     sadness       0.95      0.93      0.94       581
         joy       0.91      0.95      0.93       695
        love       0.80      0.80      0.80       159
       anger       0.92      0.86      0.89       275
        fear       0.89      0.86      0.87       224
    surprise       0.70      0.77      0.73        66

    accuracy                           0.90      2000
   macro avg       0.86      0.86      0.86      2000
weighted avg       0.90      0.90      0.90      2000



## 2.8 Testing CNN with new sentences

In [ ]:
def predict_text_cnn(text, model, vocab, max_len, label_names, device):
    model.eval()
    text = clean_text(text)
    encoded = encode_text(text, vocab)
    padded = pad_sequence(encoded, max_len)

    x = torch.tensor([padded], dtype=torch.long).to(device)
    with torch.no_grad():
        logits = model(x)
        pred = torch.argmax(logits, dim=1).item()

    return label_names[pred]

samples = [
    "I am feeling great and excited today",
    "I am really scared about tomorrow",
    "I miss my friends and feel lonely"
]

for s in samples:
    print(s, "->", predict_text_cnn(s, cnn_model, vocab, max_len, label_names, device))

I am feeling great and excited today -> joy
I am really scared about tomorrow -> fear
I miss my friends and feel lonely -> sadness


# 3. Transformer approach (DistilBERT)

Fine-tune DistilBERT for emotion classification.
- checkpoint: `distilbert-base-uncased`

## 3.1 Tokenizer và tokenize dataset

In [ ]:
checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example["text"], truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_dataset

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})

## 3.2 Metric for Transformer

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")

    return {
        "accuracy": acc["accuracy"],
        "f1_macro": f1["f1"]
    }

## 3.3 Load model

In [ ]:
id2label = {i: label_names[i] for i in range(num_classes)}
label2id = {label_names[i]: i for i in range(num_classes)}

transformer_model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 3.4 Configuration for Training

In [ ]:
training_args = TrainingArguments(
    output_dir="./emotion_distilbert",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    save_total_limit=1
)

## 3.5 Train Transformer

In [ ]:
trainer = Trainer(
    model=transformer_model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

start_time_transformer = time.time()
trainer.train()
transformer_train_time = time.time() - start_time_transformer

print(f"\nTransformer training time: {transformer_train_time:.2f} seconds")

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.483977,0.220644,0.929500,0.904519
2,0.156119,0.165733,0.933500,0.906853
3,0.104983,0.149450,0.942500,0.920655


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Transformer training time: 304.93 seconds


## 3.6 Evaluation on test set

In [ ]:
transformer_test_results = trainer.evaluate(tokenized_dataset["test"])
transformer_test_results

{'eval_loss': 0.17541250586509705,
 'eval_accuracy': 0.926,
 'eval_f1_macro': 0.8817388354675307,
 'eval_runtime': 3.398,
 'eval_samples_per_second': 588.577,
 'eval_steps_per_second': 36.786,
 'epoch': 3.0}

In [ ]:
pred_output = trainer.predict(tokenized_dataset["test"])
transformer_y_pred = np.argmax(pred_output.predictions, axis=-1)
transformer_y_true = pred_output.label_ids

transformer_test_acc = accuracy_score(transformer_y_true, transformer_y_pred)
transformer_test_f1 = f1_score(transformer_y_true, transformer_y_pred, average="macro")

print("Transformer Test accuracy:", round(transformer_test_acc, 4))
print("Transformer Test macro F1:", round(transformer_test_f1, 4))

Transformer Test accuracy: 0.926
Transformer Test macro F1: 0.8817


In [ ]:
print(classification_report(transformer_y_true, transformer_y_pred, target_names=label_names))

              precision    recall  f1-score   support

     sadness       0.95      0.97      0.96       581
         joy       0.95      0.94      0.94       695
        love       0.82      0.83      0.82       159
       anger       0.94      0.90      0.92       275
        fear       0.87      0.94      0.90       224
    surprise       0.84      0.65      0.74        66

    accuracy                           0.93      2000
   macro avg       0.90      0.87      0.88      2000
weighted avg       0.93      0.93      0.93      2000



## 3.7 Validation for Transformer with new sentences

In [ ]:
transformer_model = trainer.model
transformer_model.to(device)

def predict_text_transformer(text, model, tokenizer, device):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        pred_id = torch.argmax(outputs.logits, dim=1).item()

    return model.config.id2label[pred_id]

samples = [
    "I am feeling great and excited today",
    "I am really scared about tomorrow",
    "I miss my friends and feel lonely"
]

for s in samples:
    print(s, "->", predict_text_transformer(s, transformer_model, tokenizer, device))

I am feeling great and excited today -> joy
I am really scared about tomorrow -> fear
I miss my friends and feel lonely -> sadness


# 4. CNN vs Transformer

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Model": "CNN",
        "Test Accuracy": cnn_test_acc,
        "Test Macro F1": cnn_test_f1,
        "Training Time (sec)": cnn_train_time
    },
    {
        "Model": "DistilBERT",
        "Test Accuracy": transformer_test_acc,
        "Test Macro F1": transformer_test_f1,
        "Training Time (sec)": transformer_train_time
    }
])

comparison_df

,Model,Test Accuracy,Test Macro F1,Training Time (sec)
0,CNN,0.9025,0.860529,12.044697
1,DistilBERT,0.9260,0.881739,304.927485


In [ ]:
winner_acc = comparison_df.loc[comparison_df["Test Accuracy"].idxmax(), "Model"]
winner_f1 = comparison_df.loc[comparison_df["Test Macro F1"].idxmax(), "Model"]
faster_model = comparison_df.loc[comparison_df["Training Time (sec)"].idxmin(), "Model"]

print("Best model by Accuracy:", winner_acc)
print("Best model by Macro F1:", winner_f1)
print("Faster model:", faster_model)

Best model by Accuracy: DistilBERT
Best model by Macro F1: DistilBERT
Faster model: CNN


## Conclusion:


# 5. Trying with your own sentences

In [ ]:
my_text = "I am so happy because I got good results today"

print("CNN prediction:", predict_text_cnn(my_text, cnn_model, vocab, max_len, label_names, device))
print("Transformer prediction:", predict_text_transformer(my_text, transformer_model, tokenizer, device))

CNN prediction: joy
Transformer prediction: joy
